# WAFT-Stereo MobileNetV4 Kaggle Training Notebook
This notebook provides a complete environment and automated pipeline to run **WAFT-Stereo** model training and evaluations on Kaggle. It includes automated preflight checks, dataset checks, cloud recovery using Cloudflare R2 bucket credentials, training execution, and evaluation statistics.

## Environment Setup:
Before running this notebook, add the following parameters as **Kaggle User Secrets** (Go to *Add-ons -> Secrets* in Kaggle editor):
1. `CF_R2_ACCESS_KEY_ID`
2. `CF_R2_SECRET_ACCESS_KEY`
3. `CF_R2_ENDPOINT_URL` (Format: `https://<account-id>.r2.cloudflarestorage.com`)
4. `CF_R2_BUCKET_NAME`
5. `WANDB_API_KEY` (Optional, to sync training stats to Weights & Biases)

## 1. Credentials Setup & Package Installation
We load secret credentials from Kaggle User Secrets, write them to a local `.env` file, and install required libraries.

In [ ]:
# Load secrets dynamically on Kaggle
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    
    keys = [
        'CF_R2_ACCESS_KEY_ID', 
        'CF_R2_SECRET_ACCESS_KEY', 
        'CF_R2_ENDPOINT_URL', 
        'CF_R2_BUCKET_NAME',
        'WANDB_API_KEY'
    ]
    
    # Write to local .env
    with open('.env', 'w') as f:
        for key in keys:
            try:
                val = user_secrets.get_secret(key)
                f.write(f"{key}={val}\n")
                # Set env variable locally too
                import os
                os.environ[key] = val
            except:
                print(f"Warning: Secret '{key}' not configured.")
    print(".env file generated successfully from Kaggle Secrets!")
except Exception as e:
    print(f"Running locally or UserSecretsClient failed. Using existing .env: {e}")

# Install dependencies
!pip install -q timm peft einops boto3 python-dotenv wandb

## 2. Preflight Check
Verify GPU hardware availability, CUDA libraries, and python dependencies.

In [ ]:
import torch
import os
import sys

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device name:", torch.cuda.get_device_name(0))
    print("Allocated CUDA memory:", torch.cuda.memory_allocated(0))
else:
    print("WARNING: No GPU detected. Training will be extremely slow or fail on CPU!")

## 3. Dataset Availability Check
Verify directories and file formats for both Sanpo-Synthetic and Sanpo-Real datasets.

In [ ]:
import glob

def check_dataset(name, root):
    print(f"Checking dataset '{name}' in path: {root}...")
    if not os.path.exists(root):
        print(f"  [MISSING] Folder {root} does not exist!")
        return
    
    sessions = glob.glob(os.path.join(root, 'session_*'))
    print(f"  Found {len(sessions)} session folders.")
    
    if len(sessions) > 0:
        first_sess = sorted(sessions)[0]
        left_imgs = glob.glob(os.path.join(first_sess, 'left', '*.png'))
        right_imgs = glob.glob(os.path.join(first_sess, 'right', '*.png'))
        
        # Check depth format depending on type
        if 'synthetic' in name:
            depths = glob.glob(os.path.join(first_sess, 'depth', '*.npy'))
        else:
            depths = glob.glob(os.path.join(first_sess, 'depth_ml', '*.npy'))
            
        calib_file = os.path.join(first_sess, 'calib.json')
        
        print(f"  First Session statistics ({os.path.basename(first_sess)}):")
        print(f"    Left Images: {len(left_imgs)}")
        print(f"    Right Images: {len(right_imgs)}")
        print(f"    Depth Maps: {len(depths)}")
        print(f"    Calib JSON file exists: {os.path.exists(calib_file)}")
        
        if len(left_imgs) == 0 or len(depths) == 0 or not os.path.exists(calib_file):
            print("    [ERROR] Missing expected files inside session!")
        else:
            print("    [SUCCESS] Layout verified successfully.")

# Adjust paths if mapping to Kaggle inputs (e.g. /kaggle/input/sanpo-dataset/...)
check_dataset("sanpo_synthetic", "datasets/sanpo_synthetic")
check_dataset("sanpo_real", "datasets/sanpo_real")

## 4. Training Recovery Check
Authenticate with Cloudflare R2 bucket and check if a previous run's `latest` checkpoint is available to resume.

In [ ]:
import dotenv
import boto3
dotenv.load_dotenv()

access_key = os.getenv("CF_R2_ACCESS_KEY_ID")
secret_key = os.getenv("CF_R2_SECRET_ACCESS_KEY")
endpoint = os.getenv("CF_R2_ENDPOINT_URL")
bucket_name = os.getenv("CF_R2_BUCKET_NAME")

if not all([access_key, secret_key, endpoint, bucket_name]):
    print("WARNING: R2 Credentials are not configured. Cloud recovery and checkpoint uploads will be skipped.")
else:
    try:
        s3 = boto3.client(
            's3',
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key,
            endpoint_url=endpoint
        )
        # List contents of the bucket
        res = s3.list_objects_v2(Bucket=bucket_name)
        print(f"Connected to Cloudflare R2 bucket '{bucket_name}' successfully!")
        print("Available cloud checkpoints:")
        if 'Contents' in res:
            for obj in res['Contents']:
                if '.pth' in obj['Key']:
                    print(f"  - {obj['Key']} ({obj['Size'] / 1e6:.2f} MB, modified: {obj['LastModified']})")
        else:
            print("  None found.")
    except Exception as e:
        print(f"Failed to connect to R2 bucket: {e}")

## 5. Start Training
Run the training loop on Kaggle. The script will automatically download the cloud checkpoint (if available) and continue training, saving the best models to the cloud every 50 iterations when updated.

In [ ]:
# Execute Stage 1 (Sanpo Synthetic) or Stage 2 (Sanpo Real) training
# Override configurations at command line if needed, e.g. SOLVER.IMS_PER_BATCH 4
!python main.py --config-file configs/custom/stage1-sanpo-synthetic.yaml --num-gpus 1

## 6. Evaluation
Evaluate the model against test datasets to compute EPE (End-point Error) and D1 statistics.

In [ ]:
# Evaluate using checkpoint_best.pth or latest
!python main.py --config-file configs/custom/stage1-sanpo-synthetic.yaml --eval-only --ckpt ckpts/custom/stage1-sanpo-synthetic/checkpoint_best.pth